# 🧬 ColabPMI: Protein-Molecule Interaction Prediction Platform

**ColabPMI** is a comprehensive platform for protein-molecule interaction prediction and model training, running directly in Google Colab.

# How to start (🎥 [tutorial](https://www.youtube.com/watch?v=nmLtjlCI_7M))

## Follow below steps to switch to GPU. Once connected, click the run-button <img src='https://github.com/westlake-repl/SaProtHub/blob/dev/Figure/run_button.png?raw=true' height='25px' width='25px' align='center'> to run ColabPMI

<img src="https://github.com/westlake-repl/SaProtHub/blob/main/Figure/Switch_Runtime_2.png?raw=true" align="center">

#Notice:

1.Please connect to GPU before clicking on the run button.(required)

T4 GPU: Free to use. Suitable for low-compute tasks. Note that this free GPU is unstable, so you may frequently lose connection or have memory issues during training and prediction. To avoid this problem you could switch to L4 or A100 GPU by subscribing Colab Pro.

L4 GPU (need Colab Pro): It has larger memory.

A100 GPU (need Colab Pro): Much more powerful GPU. Since no 650M model is provided in the program,T4 GPU is enough for training and prediction if the average length of protein sequence is around 800. L4 GPU and A100 GPU can be used for larger sequence.

2.If you want your trained model to be saved after the runtime is disconncted, please click "Allow" to grant Colab access to your Google Drive after the program is started.(optional)

**Contact {caixiaoyao, sujin, yuanfajie}@westlake.edu.cn for questions**

In [ ]:
# @title

import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    """固定所有随机种子以确保完全可重复性"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # 关键：禁用 CuDNN 的非确定性优化
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ['PYTHONHASHSEED'] = str(seed)
   # print(f"✓ Fixed all random seeds to {seed}")

# 立即设置种子
set_seed(42)

# ==================== 导入所有库 ====================
!pip install 'jupyter_ui_poll'
!pip install 'loguru'
!pip install 'colorama'

# Suppress HF token warning
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='huggingface_hub')

from datetime import datetime
import glob
import io
import re
import sys
import time

import shutil
import json
import threading
import traceback
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import requests
import base64
import gc
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, FileLink
from tqdm.notebook import tqdm
from functools import partial
import queue
from datetime import datetime
from jupyter_ui_poll import ui_events
import markdown
from loguru import logger
from easydict import EasyDict
from colorama import init, Fore, Back, Style
from safetensors import safe_open
from huggingface_hub import snapshot_download, hf_hub_download

# Verify imports
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"PyTorch version: {torch.__version__}")

# Suppress HuggingFace logging
hf_logging.set_verbosity_error()

try:
    from google.colab import files, drive, output
    IN_COLAB = True
except:
    IN_COLAB = False

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path setup
root_dir = os.getcwd()

# Try to install SaprotHub if needed (optional)
try:
    sys.path.append(f"{root_dir}/SaprotHub")
    DATASET_HOME = f'{root_dir}/SaprotHub/datasets'
    ADAPTER_HOME = f'{root_dir}/SaprotHub/adapters'
    OUTPUT_HOME = f'{root_dir}/SaprotHub/output'
    UPLOAD_FILE_HOME = f'{root_dir}/SaprotHub/upload_files'
except:
    # If SaprotHub not installed, use local paths
    DATASET_HOME = f'{root_dir}/datasets'
    ADAPTER_HOME = f'{root_dir}/adapters'
    OUTPUT_HOME = f'{root_dir}/output'
    UPLOAD_FILE_HOME = f'{root_dir}/upload_files'

CHECKPOINT_DIR = f'{root_dir}/checkpoints'

for path in [DATASET_HOME, ADAPTER_HOME, OUTPUT_HOME, UPLOAD_FILE_HOME, CHECKPOINT_DIR]:
    os.makedirs(path, exist_ok=True)

if IN_COLAB:
    try:
        drive.mount('/content/drive')
        CHECKPOINT_DIR = '/content/drive/MyDrive/SaprotHub_checkpoints/'
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        print("Google Drive mounted")
    except:
        pass

# Color class using colorama
init(autoreset=True)

class Font:
    RED = Fore.RED
    GREEN = Fore.GREEN
    YELLOW = Fore.YELLOW
    BLUE = Fore.BLUE
    BOLD = Style.BRIGHT
    RESET = Style.RESET_ALL

# Task type dictionary
task_type_dict = {
    "Regression": "regression",
    "Molecule-Protein Interaction": "interaction"
}

# Model type dictionary
model_type_dict = {
    "regression": "saprot/saprot_regression_model",
    "interaction": None
}

print("✅ Setup complete")

# ==================== 模型定义（确定性版本） ====================
class MyModel(nn.Module):
    """回归模型：使用Cross-Attention学习蛋白质-药物交互，输出乘以1e4"""
    def __init__(self, pro_dim=480, drug_dim=384, hidden_dim=256, dropout=0.2):
        super().__init__()

        # 蛋白质投影
        self.proj_pro = nn.Sequential(
            nn.Linear(pro_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # 药物投影
        self.proj_drug = nn.Sequential(
            nn.Linear(drug_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Cross-Attention 层
        self.cross_attn = nn.MultiheadAttention(
            hidden_dim, num_heads=4, batch_first=True, dropout=0.1
        )

        # 交互网络
        self.interaction_net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
        )

        # 回归头
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.attention_weights = None

    def forward(self, pro, drug):
        pro = self.proj_pro(pro).unsqueeze(1)   # [batch, 1, hidden_dim]
        drug = self.proj_drug(drug).unsqueeze(1) # [batch, 1, hidden_dim]

        # Cross-Attention
        drug_to_protein, attn1 = self.cross_attn(drug, pro, pro)
        protein_to_drug, attn2 = self.cross_attn(pro, drug, drug)

        self.attention_weights = {'drug_to_protein': attn1, 'protein_to_drug': attn2}

        # 拼接交互特征
        interaction = torch.cat([drug_to_protein.squeeze(1),
                                 protein_to_drug.squeeze(1)], dim=1)

        interaction_feat = self.interaction_net(interaction)
        out = self.regressor(interaction_feat)
        return out.squeeze()

# Molecule-Protein Interaction Model (确定性版本)
class InteractionModel(nn.Module):
    """使用点积的蛋白质-药物交互分类模型 - 确定性版本"""
    def __init__(self, pro_dim=480, drug_dim=384, hidden_dim=256, dropout=0.2):
        super().__init__()

        # 蛋白质投影
        self.protein_proj = nn.Sequential(
            nn.Linear(pro_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # 药物投影
        self.drug_proj = nn.Sequential(
            nn.Linear(drug_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # 可学习的温度和偏置
        self.temperature = nn.Parameter(torch.ones(1) * 0.5)
        self.bias = nn.Parameter(torch.zeros(1))

        # 调试标志
        self.debug = False

    def forward(self, pro, drug):
        # 投影到共享空间
        pro_emb = self.protein_proj(pro)
        drug_emb = self.drug_proj(drug)

        if self.debug:
            print(f"[DEBUG] pro_emb: mean={pro_emb.mean().item():.6f}, std={pro_emb.std().item():.6f}")
            print(f"[DEBUG] drug_emb: mean={drug_emb.mean().item():.6f}, std={drug_emb.std().item():.6f}")

        # L2归一化（关键：让点积范围在[-1,1]，跨药物可比）
        pro_emb = F.normalize(pro_emb, p=2, dim=1)
        drug_emb = F.normalize(drug_emb, p=2, dim=1)

        # 点积交互
        dot_product = (pro_emb * drug_emb).sum(dim=1, keepdim=True)

        # 温度缩放 + 偏置
        temp = self.temperature.abs() + 1e-8
        logit = dot_product / temp + self.bias

        # 直接输出0-1概率
        prob = torch.sigmoid(logit)



        return prob.squeeze(dim=1)

# Dataset Class
class InteractionDataset(Dataset):
    def __init__(self, protein_features, drug_features, labels):
        self.protein_features = protein_features
        self.drug_features = drug_features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'protein': self.protein_features[idx],
            'drug': self.drug_features[idx],
            'label': self.labels[idx]
        }

def file_split(df):
    if 'split' not in df.columns and 'state' not in df.columns and 'subset' not in df.columns and 'set' not in df.columns:
        train_val_data, test_data = train_test_split(
            df,
            test_size=1/9,
            random_state=20,
        )
        train_data, val_data = train_test_split(
            train_val_data,
            test_size=1/8,
            random_state=20,
        )
    else:
        if 'split' in df.columns:
            split_col = 'split'
        elif 'stage' in df.columns:
            split_col = 'stage'
        elif 'set' in df.columns:
            split_col = 'set'
        else:
            split_col = 'subset'
        df[split_col] = df[split_col].astype(str).str.lower().str.strip()
        df[split_col] = df[split_col].replace(['training', 'trn', 't'], 'train')
        df[split_col] = df[split_col].replace(['val', 'valid', 'vld', 'validation_set', 'dev'], 'validation')
        df[split_col] = df[split_col].replace(['testing', 'tst', 'te'], 'test')
        df.columns = [col.lower().strip() for col in df.columns]
        column_mapping = {
            'protein': 'seq', 'sequence': 'seq', 'protein_seq': 'seq', 'target_seq': 'seq', 'seqs': 'seq',
            'drug': 'smiles', 'molecule': 'smiles', 'compound': 'smiles', 'canonical_smiles': 'smiles',
            'labels': 'label', 'active': 'label', 'activity': 'label', 'y': 'label', 'bind': 'label'
        }
        df = df.rename(columns=column_mapping)
        train_data = df[df[split_col] == 'train'].copy()
        val_data = df[df[split_col] == 'validation'].copy()
        test_data = df[df[split_col] == 'test'].copy()
    if len(val_data) == 0:
        train_data, val_data = train_test_split(
            train_data,
            test_size=1/8,
            random_state=20
        )
    if len(test_data) == 0:
        train_data, test_data = train_test_split(
            train_data,
            test_size=1/8,
            random_state=20
        )
    return train_data, val_data, test_data

# ==================== 确定性特征提取函数 ====================
def extract_protein_features_deterministic(sequences, tokenizer, model, batch_size=8):
    """确定性版本的特征提取 - 使用 inference_mode"""

    all_features = []
    model.eval()

    # 自动检测编码器类型（通过模型名称）
    encoder_type = 'unknown'
    if hasattr(model, 'config'):
        model_name = getattr(model.config, '_name_or_path', '').lower()
        if 'saprot' in model_name:
            encoder_type = 'saprot'
        elif 'esm' in model_name:
            encoder_type = 'esm2'

    # 如果从config没检测到，尝试从tokenizer检测
    if encoder_type == 'unknown':
        tokenizer_name = getattr(tokenizer, 'name_or_path', '').lower()
        if 'saprot' in tokenizer_name:
            encoder_type = 'saprot'
        elif 'esm' in tokenizer_name:
            encoder_type = 'esm2'

    # 打印检测结果，方便用户确认
    # print(f"   🔍 Detected encoder: {encoder_type.upper()}")

    # 批量处理
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]

        # 根据编码器类型处理序列
        processed = []
        for seq in batch:
            # 统一处理特殊氨基酸
            seq_clean = re.sub(r'[UZOB]', '#', seq)

            if encoder_type == 'saprot':
                # SaProt: 使用 '#' 分隔
                processed_seq = '#'.join(list(seq_clean))
            elif encoder_type == 'esm2':
                # ESM: 使用空格分隔
                processed_seq = ' '.join(list(seq_clean))
            else:
                # 未知类型：尝试原始序列（让tokenizer自己处理）
                processed_seq = seq_clean
                print(f"   ⚠️ Unknown encoder type, using raw sequence")

            processed.append(processed_seq)

        # Tokenize
        inputs = tokenizer(processed, padding='max_length', truncation=True,
                          max_length=512, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # 推理
        with torch.inference_mode():  # 使用 inference_mode 比 no_grad 更快
            outputs = model(**inputs)
            all_features.append(outputs.last_hidden_state[:, 0, :].cpu())

        # 清理缓存
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

    return torch.cat(all_features, dim=0) if all_features else torch.tensor([])

def extract_drug_features_deterministic(smiles_list, tokenizer, model, batch_size=8):
    """确定性版本的特征提取 - 使用 inference_mode"""
    all_features = []
    model.eval()

    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        inputs = tokenizer(batch, padding='max_length', truncation=True,
                          max_length=512, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # 使用 inference_mode 替代 no_grad
        with torch.inference_mode():
            outputs = model(**inputs)
            all_features.append(outputs.last_hidden_state[:, 0, :].cpu())

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

    return torch.cat(all_features, dim=0) if all_features else torch.tensor([])

# 保留原始函数用于训练（训练不需要确定性）
extract_protein_features = extract_protein_features_deterministic
extract_drug_features = extract_drug_features_deterministic

class TrainingController:
    def __init__(self):
        self.stop_requested = False
        self.pause_event = threading.Event()
        self.pause_event.set()
        self.thread = None
        self.now_thread = None
        self.is_training_finished = False
        self.current_model = None
        self.tokenizer_pro = None
        self.model_pro = None
        self.tokenizer_drug = None
        self.model_drug = None
        self.pro_dim = None
        self.drug_dim = None
        self.current_regression_model = None
        self.regression_model_name = None
        self.task_type = 'interaction'

    def reset(self):
        self.stop_requested = False
        self.pause_event.set()
        self.is_training_finished = False
        self.thread = None
        self.now_thread = None

# Global controller
ctrl = TrainingController()
now_thread = None
refresh_module = None
main_display = widgets.Output()
train_log_output = widgets.Output()

# System buttons
HINT = HTML(markdown.markdown("**<font color=red>Hint: You can click the buttons below to stop or restart at any time</font>"))
stop_btn = widgets.Button(description="Stop", button_style="danger")
back_btn = widgets.Button(description="Back", button_style="success")
refresh_btn = widgets.Button(description="Refresh", button_style="success")
system_hint = HTML(markdown.markdown(
    "**Back:** Stop current task and return to main menu<br>"
    "**Refresh**: Stop current task and refresh current interface<br>"
    "**Stop**: Stop current task"
))
system_widgets = [HINT, widgets.HBox([back_btn, refresh_btn, stop_btn]), system_hint]

def stop_thread(button):
    global now_thread
    if now_thread and now_thread.is_alive():
        print(f"{Font.RED}Interrupting running task...{Font.RESET}")
        import ctypes
        import inspect

        while now_thread.is_alive():
            exctype = SystemExit
            tid = ctypes.c_long(now_thread.ident)
            if not inspect.isclass(exctype):
                exctype = type(exctype)
            res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
            if res == 0:
                continue
            elif res != 1:
                ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
                continue

        print(f"{Font.RED}\nTask interrupted{Font.RESET}")
        now_thread = None
        ctrl.now_thread = None
    else:
        print(f"{Font.RED}\nNo running task{Font.RESET}")

def start_thread(func, args=tuple()):
    def wrapper(*args, **kwargs):
        try:
            func(*args, **kwargs)
        except Exception as e:
            print(f"{Font.RED}Thread error: {e}{Font.RESET}")
            traceback.print_exc()

    thread = threading.Thread(target=wrapper, args=args)
    thread.daemon = True
    thread.start()

    global now_thread
    now_thread = thread
    ctrl.now_thread = thread
    return thread

def jump(button, next=None):
    stop_thread(None)
    clear_output(wait=True)

    if next is None:
        if refresh_module:
            refresh_module()
    else:
        next()

    display(main_display)
    display(*system_widgets)

def custom_display(*args, **kwargs):
    clear_output()
    display(*args, **kwargs)
    display(*system_widgets)

# ==================== 训练函数 ====================
def train_interaction(df, params, task_type):
    """Molecule-protein interaction training"""
    global ctrl
    ctrl.task_type = task_type
    try:
        with train_log_output:
            print("=" * 60)
            print("Starting training")
            print("=" * 60)

            # 1. Load encoders
            print("\n1. Loading encoders...")
            tokenizer_pro = AutoTokenizer.from_pretrained(params['pro_enc'])
            model_pro = AutoModel.from_pretrained(params['pro_enc']).to(device)
            tokenizer_drug = AutoTokenizer.from_pretrained(params['drug_enc'])
            model_drug = AutoModel.from_pretrained(params['drug_enc']).to(device)

            ctrl.tokenizer_pro = tokenizer_pro
            ctrl.model_pro = model_pro
            ctrl.tokenizer_drug = tokenizer_drug
            ctrl.model_drug = model_drug

            # 2. Data splitting
            print("\n2. Data splitting...")
            train, val, test = file_split(df)

            print(f"   Training set: {len(train)} samples")
            print(f"   Validation set: {len(val)} samples")
            print(f"   Test set: {len(test)} samples")

            if ctrl.stop_requested:
                return

            # 3. Feature extraction
            print("\n3. Feature extraction...")
            with tqdm(total=4, desc="Extracting features") as pbar:
                pro_train = extract_protein_features(train['seq'].tolist(), tokenizer_pro, model_pro)
                pbar.update(1)
                if ctrl.stop_requested: return

                drug_train = extract_drug_features(train['smiles'].tolist(), tokenizer_drug, model_drug)
                pbar.update(1)
                if ctrl.stop_requested: return

                pro_val = extract_protein_features(val['seq'].tolist(), tokenizer_pro, model_pro)
                pbar.update(1)
                if ctrl.stop_requested: return

                drug_val = extract_drug_features(val['smiles'].tolist(), tokenizer_drug, model_drug)
                pbar.update(1)

            if ctrl.stop_requested:
                return

            # 4. Initialize model
            print("\n4. Initializing model...")
            url = ('https://raw.githubusercontent.com/caixiaoyao2025/scientific_training1_cxy/main/simpledotproductclassifier.pth'
                   if task_type == 'interaction'
                   else 'https://raw.githubusercontent.com/caixiaoyao2025/scientific_training1_cxy/main/regression_model.pth')
            response = requests.get(url)
            response = io.BytesIO(response.content)
            data = torch.load(response, map_location='cpu', weights_only=False)

            if task_type == 'interaction':
                ctrl.current_model = InteractionModel(data['model_config']['pro_dim'],
                                                       data['model_config']['drug_dim']).to(device)
            else:
                ctrl.current_model = MyModel(data['model_config']['pro_dim'],
                                              data['model_config']['drug_dim']).to(device)

            model = ctrl.current_model
            model.load_state_dict(data['model_state_dict'])
            optimizer = optim.Adam(model.parameters(), lr=params['lr'])
            criterion = nn.BCELoss() if ctrl.task_type == 'interaction' else nn.MSELoss()

            train_dataset = InteractionDataset(pro_train, drug_train,
                                              torch.tensor(train['label'].values).float())
            val_dataset = InteractionDataset(pro_val, drug_val,
                                            torch.tensor(val['label'].values).float())

            train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=params['batch_size'])

            # 5. Training loop
            print("\n5. Starting training...")
            print("-" * 60)

            best_val_loss = float('inf')
            patience_counter = 0

            for epoch in range(params['epochs']):
                if ctrl.stop_requested:
                    break

                # Training
                model.train()
                train_loss = 0
                for batch in train_loader:
                    if ctrl.stop_requested:
                        break

                    p = batch['protein'].to(device)
                    d = batch['drug'].to(device)
                    l = batch['label'].to(device).float()

                    optimizer.zero_grad()
                    out = model(p, d)
                    loss = criterion(out, l)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    train_loss += loss.item()

                if ctrl.stop_requested:
                    break

                # Validation
                model.eval()
                val_loss = 0
                with torch.no_grad():
                    for batch in val_loader:
                        if ctrl.stop_requested:
                            break
                        p = batch['protein'].to(device)
                        d = batch['drug'].to(device)
                        l = batch['label'].to(device).float()
                        out = model(p, d)
                        val_loss += criterion(out, l)
                avg_val_loss = val_loss / len(val_loader)
                print(f"Epoch {epoch+1:2d}/{params['epochs']} | Val Loss: {avg_val_loss:.6f}")

                # Save checkpoint
                checkpoint = {
                    'epoch': epoch+1,
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'best_val_loss': best_val_loss,
                    'pro_dim': 480,
                    'drug_dim': 384
                }
                torch.save(checkpoint, f"{CHECKPOINT_DIR}/last.pth.tar")

                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    patience_counter = 0
                    torch.save(checkpoint, f"{CHECKPOINT_DIR}/best.pth.tar")
                else:
                    patience_counter += 1

                if patience_counter >= params['patience']:
                    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                    filename = f"{timestamp}"
                    if task_type == 'interaction':
                        torch.save(checkpoint, f"{CHECKPOINT_DIR}/classification_best_{filename}.pth.tar")
                    else:
                        torch.save(checkpoint, f"{CHECKPOINT_DIR}/regression_best_{filename}.pth.tar")
                    break

                # Clean cache
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    gc.collect()

            if not ctrl.stop_requested:
                print("\n" + "=" * 60)
                print("✅ Training complete!")
                print("Please go to the top of the output area if you don't see the 'start prediction' button.")
                print("=" * 60)
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"{timestamp}"
                if task_type == 'interaction':
                    torch.save(checkpoint, f"{CHECKPOINT_DIR}/classification_best_{filename}.pth.tar")
                else:
                    torch.save(checkpoint, f"{CHECKPOINT_DIR}/regression_best_{filename}.pth.tar")
                ctrl.is_training_finished = True

    except Exception as e:
        with train_log_output:
            print(f"\n❌ Error: {e}")
            traceback.print_exc()

# ==================== 确定性预测函数 ====================
def load_prediction_model_deterministic(task_type, device):
    """确定性版本的模型加载"""
    if task_type == 'interaction':
        url = 'https://raw.githubusercontent.com/caixiaoyao2025/scientific_training1_cxy/main/simpledotproductclassifier.pth'
    else:
        url = 'https://raw.githubusercontent.com/caixiaoyao2025/scientific_training1_cxy/main/regression_model.pth'

    response = requests.get(url)
    data = torch.load(io.BytesIO(response.content), map_location='cpu', weights_only=False)

    if task_type == 'regression':
        model = MyModel(data['model_config']['pro_dim'], data['model_config']['drug_dim'])
    else:
        model = InteractionModel(data['model_config']['pro_dim'], data['model_config']['drug_dim'])

    # 使用 strict=False 加载
    missing_keys, unexpected_keys = model.load_state_dict(data['model_state_dict'], strict=False)

    if missing_keys:
        print(f"   ⚠️ Missing keys: {missing_keys}")
    if unexpected_keys:
        print(f"   ⚠️ Unexpected keys: {unexpected_keys}")

    model = model.to(device)
    model.eval()

    # 确保所有子模块也是 eval 模式
    for module in model.modules():
        if isinstance(module, (nn.Dropout, nn.BatchNorm1d)):
            module.eval()
    return model

def run_deterministic_prediction(pro_sequences, drug_smiles, regression_model,
                                  tokenizer_pro, model_pro,
                                  tokenizer_drug, model_drug,
                                  task_type='regression'):
    """端到端预测：使用批量特征提取"""
    device = next(regression_model.parameters()).device

    # 确保输入是列表
    if isinstance(pro_sequences, str):
        pro_sequences = [pro_sequences]
    if isinstance(drug_smiles, str):
        drug_smiles = [drug_smiles]

    # 1. 批量提取蛋白质特征（使用你已有的函数）
    print(f"   Extracting protein features for {len(pro_sequences)} sequences...")
    pro_features = extract_protein_features_deterministic(
        pro_sequences, tokenizer_pro, model_pro, batch_size=8
    )

    # 2. 批量提取药物特征
    print(f"   Extracting drug features for {len(drug_smiles)} molecules...")
    drug_features = extract_drug_features_deterministic(
        drug_smiles, tokenizer_drug, model_drug, batch_size=8
    )

    # 3. 预测
    pro_tensor = pro_features.to(device)
    drug_tensor = drug_features.to(device)

    regression_model.eval()
    with torch.inference_mode():
        predictions = regression_model(pro_tensor, drug_tensor)

    # 4. 返回结果
    if predictions.dim() == 0:
        return predictions.item()
    elif len(predictions) == 1:
        return predictions[0].item()
    else:
        return predictions.cpu().tolist()

# ==================== UI 函数 ====================
def show_main_menu():
    global refresh_module
    refresh_module = show_main_menu
    ctrl.reset()

    with main_display:
        clear_output()

        title = widgets.HTML("<h2 style='color: #2c3e50; text-align: center;'>Molecule-Protein Interaction Prediction</h2>")

        btn_predict = widgets.Button(
            description="🔮 Start prediction directly",
            button_style='info',
            layout=widgets.Layout(width='300px', height='40px', margin='5px')
        )
        btn_train = widgets.Button(
            description="⚙️ Train Model",
            button_style='primary',
            layout=widgets.Layout(width='300px', height='40px', margin='5px')
        )

        btn_predict.on_click(lambda b: jump(b, next=show_prediction_selection))
        btn_train.on_click(lambda b: jump(b, next=show_training_task_selection))

        display(widgets.VBox([title, btn_predict, btn_train],
                            layout=widgets.Layout(align_items='center', padding='20px')))

def show_prediction_selection():
    global refresh_module, ctrl
    refresh_module = show_prediction_selection

    with main_display:
        clear_output()

        title = widgets.HTML("<h3>⚙️ Select Prediction Task</h3>")

        btn_regression = widgets.Button(
            description="📊 Regression Prediction",
            button_style='info',
            layout=widgets.Layout(width='400px', height='40px', margin='5px')
        )
        btn_interaction = widgets.Button(
            description="🧪 Classification Prediction",
            button_style='primary',
            layout=widgets.Layout(width='400px', height='40px', margin='5px')
        )

        def on_reg_click(b):
            ctrl.task_type = 'regression'
            jump(b, next=show_interaction_prediction)

        def on_int_click(b):
            ctrl.task_type = 'interaction'
            jump(b, next=show_interaction_prediction)

        btn_regression.on_click(on_reg_click)
        btn_interaction.on_click(on_int_click)

        display(widgets.VBox([
            title,
            btn_regression,
            btn_interaction,
        ], layout=widgets.Layout(align_items='center', padding='20px')))

def show_training_task_selection():
    global refresh_module
    refresh_module = show_training_task_selection

    with main_display:
        clear_output()

        title = widgets.HTML("<h3>⚙️ Select Training Task</h3>")

        btn_regression = widgets.Button(
            description="📊 Regression Training",
            button_style='info',
            layout=widgets.Layout(width='400px', height='40px', margin='5px')
        )
        btn_interaction = widgets.Button(
            description="🧪 Classification Training",
            button_style='primary',
            layout=widgets.Layout(width='400px', height='40px', margin='5px')
        )

        regression_intro = widgets.HTML(
            "<p style='color: gray;'>Fine-tune SaprotHub models for regression tasks (fluorescent, stability)</p>"
        )
        interaction_intro = widgets.HTML(
            "<p style='color: gray;'>Train custom classification task prediction model</p>"
        )

        btn_regression.on_click(lambda b: jump(b, next=lambda: show_regression_setup()))
        btn_interaction.on_click(lambda b: jump(b, next=lambda: show_interaction_setup()))

        display(widgets.VBox([
            title,
            btn_regression,
            btn_interaction,
        ], layout=widgets.Layout(align_items='center', padding='20px')))

def show_regression_setup():
    global refresh_module
    refresh_module = show_regression_setup

    with main_display:
        clear_output()

        title = widgets.HTML("<h3>🧪 Regression Task Training Setup</h3>")

        # Protein encoder selection
        pro_enc = widgets.Dropdown(
            options=[
                ("westlake-repl/SaProt_35M_AF2_seqOnly", "westlake-repl/SaProt_35M_AF2_seqOnly"),
                ("facebook/esm2_t12_35M_UR50D - ESM2-35M", "facebook/esm2_t12_35M_UR50D")
            ],
            value="westlake-repl/SaProt_35M_AF2_seqOnly",
            description="Protein Encoder:",
            layout=widgets.Layout(width='500px'),
            style={'description_width': 'initial'}
        )

        # Molecule encoder selection
        drug_enc = widgets.Dropdown(
            options=[
                ("DeepChem/ChemBERTa-77M-MLM", "DeepChem/ChemBERTa-77M-MLM"),
                ("DeepChem/ChemBERTa-77M-MTR", "DeepChem/ChemBERTa-77M-MTR")
            ],
            value="DeepChem/ChemBERTa-77M-MLM",
            description="Molecule Encoder:",
            layout=widgets.Layout(width='500px'),
            style={'description_width': 'initial'}
        )

        # Hyperparameters
        lr = widgets.FloatText(value=0.00001, description="Learning Rate:", layout=widgets.Layout(width='300px'))
        batch = widgets.IntText(value=32, description="Batch Size:", layout=widgets.Layout(width='300px'))
        epochs = widgets.IntText(value=30, description="Epochs:", layout=widgets.Layout(width='300px'))
        patience = widgets.IntText(value=80, description="Patience:", layout=widgets.Layout(width='300px'))

        # Data upload
        uploader = widgets.FileUpload(
            description='📤 Upload CSV Data File',
            layout=widgets.Layout(width='500px'),
            accept='.csv',
            multiple=False
        )

        # Example data info
        example_html = widgets.HTML(
            "<p style='color: gray;'>CSV file needs three columns: protein/sequence, drug/smiles, label</p>"
            "<p style='color: gray;'>If the train/val/test data is not assigned in the file, the data will be split by 8:1:1 randomly.</p>"
            "<p><a href='https://github.com/caixiaoyao2025/scientific_training1_cxy/blob/main/train_reg.csv' target='_blank'>Download example data</a></p>"
        )
        desc = widgets.HTML("<p>KIBA affinity scores combine the information from different sources, including Kd, Ki (inhibitor constant), and IC50 (the concentration required to produce half-maximum inhibition) into a single bioactivity score and are ranged from 0.0 to 17.2. The lower the KIBA score the stronger the binding affinity between the drug and the target.</p>")
        btn_start = widgets.Button(
            description="🚀 Start Training",
            button_style='success',
            layout=widgets.Layout(width='200px', margin='10px')
        )

        def start_regression(b):
            if not uploader.value:
                print("Please upload a data file first!")
                return

            content = list(uploader.value.values())[0]['content']
            df = pd.read_csv(io.BytesIO(content))

            params = {
                'pro_enc': pro_enc.value,
                'drug_enc': drug_enc.value,
                'lr': lr.value,
                'batch_size': batch.value,
                'epochs': epochs.value,
                'patience': patience.value
            }

            with main_display:
                clear_output()
            show_training_progress(df, params, 'regression')

        btn_start.on_click(start_regression)

        display(widgets.VBox([
            title,
            pro_enc,
            drug_enc,
            lr,
            batch,
            epochs,
            uploader,
            example_html,
            desc,
            btn_start
        ], layout=widgets.Layout(align_items='center', padding='20px')))

def show_interaction_setup():
    global refresh_module
    refresh_module = show_interaction_setup
    with main_display:
        clear_output()

        title = widgets.HTML("<h3>🧪 Classification Task Training Setup</h3>")

        # Protein encoder selection
        pro_enc = widgets.Dropdown(
            options=[
                ("westlake-repl/SaProt_35M_AF2_seqOnly", "westlake-repl/SaProt_35M_AF2_seqOnly"),
                ("facebook/esm2_t12_35M_UR50D - ESM2-35M", "facebook/esm2_t12_35M_UR50D")
            ],
            value="westlake-repl/SaProt_35M_AF2_seqOnly",
            description="Protein Encoder:",
            layout=widgets.Layout(width='500px'),
            style={'description_width': 'initial'}
        )

        # Molecule encoder selection
        drug_enc = widgets.Dropdown(
            options=[
                ("DeepChem/ChemBERTa-77M-MLM", "DeepChem/ChemBERTa-77M-MLM"),
                ("DeepChem/ChemBERTa-77M-MTR", "DeepChem/ChemBERTa-77M-MTR")
            ],
            value="DeepChem/ChemBERTa-77M-MLM",
            description="Molecule Encoder:",
            layout=widgets.Layout(width='500px'),
            style={'description_width': 'initial'}
        )

        # Hyperparameters
        lr = widgets.FloatText(value=0.0001, description="Learning Rate:", layout=widgets.Layout(width='300px'))
        batch = widgets.IntText(value=32, description="Batch Size:", layout=widgets.Layout(width='300px'))
        epochs = widgets.IntText(value=80, description="Epochs:", layout=widgets.Layout(width='300px'))
        patience = widgets.IntText(value=80, description="Patience:", layout=widgets.Layout(width='300px'))

        # Data upload
        uploader = widgets.FileUpload(
            description='📤 Upload CSV Data File',
            layout=widgets.Layout(width='500px'),
            accept='.csv',
            multiple=False
        )

        # Example data info
        example_html = widgets.HTML(
            "<p style='color: gray;'>CSV file needs three columns: protein/sequence, drug/smiles, label</p>"
            "<p style='color: gray;'>If the train/val/test data is not assigned in the file, it will be divided by 8:1:1 randomly.</p>"
            "<p><a href='https://github.com/caixiaoyao2025/scientific_training1_cxy/blob/main/example%20(1).csv' target='_blank'>Download example data</a></p>"
            "<p>In the example data, label=1 means the molecule-protein pair is likely to bind and label=0 means the opposite.</p>"
        )

        btn_start = widgets.Button(
            description="🚀 Start Training",
            button_style='success',
            layout=widgets.Layout(width='200px', margin='10px')
        )

        def start_interaction(b):
            if not uploader.value:
                print("Please upload a data file first!")
                return

            content = list(uploader.value.values())[0]['content']
            df = pd.read_csv(io.BytesIO(content))

            params = {
                'pro_enc': pro_enc.value,
                'drug_enc': drug_enc.value,
                'lr': lr.value,
                'batch_size': batch.value,
                'epochs': epochs.value,
                'patience': patience.value
            }

            with main_display:
                clear_output()
            show_training_progress(df, params, 'interaction')

        btn_start.on_click(start_interaction)

        display(widgets.VBox([
            title,
            pro_enc,
            drug_enc,
            lr,
            batch,
            epochs,
            uploader,
            example_html,
            btn_start
        ], layout=widgets.Layout(align_items='center', padding='20px')))

def show_training_progress(df, params, task_type='interaction'):
    global refresh_module, ctrl

    def refresh_func():
        show_training_progress(df, params, task_type)

    refresh_module = refresh_func

    if task_type == 'interaction':
        ctrl.task_type = 'interaction'
    else:
        ctrl.task_type = 'regression'

    with main_display:
        clear_output()

        status = widgets.HTML("<h3 style='color: #27ae60;'>⏳ Training in progress...</h3>")

        # Pause/Resume button
        pause_btn = widgets.Button(description="⏸️ Pause", button_style='warning', layout=widgets.Layout(width='150px'))

        def on_pause_clicked(b):
            if ctrl.pause_event.is_set():
                ctrl.pause_event.clear()
                pause_btn.description = "▶️ Resume"
                pause_btn.button_style = 'success'
                status.value = "<h3 style='color: #f39c12;'>⏸️ Training paused</h3>"
            else:
                ctrl.pause_event.set()
                pause_btn.description = "⏸️ Pause"
                pause_btn.button_style = 'warning'
                status.value = "<h3 style='color: #27ae60;'>⏳ Training in progress...</h3>"

        pause_btn.on_click(on_pause_clicked)

        display(widgets.VBox([status, widgets.HTML("<hr>"), train_log_output]))

        train_log_output.clear_output()
        ctrl.reset()

        if task_type == 'regression':
            start_thread(train_interaction, (df, params, 'regression'))
        else:
            start_thread(train_interaction, (df, params, 'interaction'))

def show_post_training_ui():
    with main_display:
        clear_output()
        train_log_output.clear_output()
        title = widgets.HTML("<h2 style='color: #27ae60; text-align: center;'>✅ Training Complete!</h2>")

        btn_predict = widgets.Button(
            description="🔮 Start Prediction",
            button_style='success',
            layout=widgets.Layout(width='200px', height='40px', margin='5px')
        )

        btn_return = widgets.Button(
            description="🏠 Return to Main Menu",
            button_style='info',
            layout=widgets.Layout(width='200px', height='40px', margin='5px')
        )
        btn_predict.on_click(lambda b: jump(b, next=show_prediction_selection))
        btn_return.on_click(lambda b: jump(b, next=show_main_menu))

        display(widgets.VBox([
            title,
            widgets.HTML("<hr>"),
            widgets.HBox([btn_predict, btn_return])
        ], layout=widgets.Layout(align_items='center', padding='20px')))

def show_interaction_prediction():
    global refresh_module, ctrl
    refresh_module = show_interaction_prediction

    with main_display:
        clear_output()

        if ctrl.task_type == 'interaction':
            title = widgets.HTML("<h2 style='color: #2c3e50;'>🧪 Classification Task Prediction</h2>")
        else:
            title = widgets.HTML("<h2 style='color: #2c3e50;'>📊 Regression Task Prediction</h2>")

        # 检查是否有训练好的模型（包括 Google Drive 中的）
        model_options = [('Use pre-trained model', 'pretrained')]

        # 1. 检查默认 Drive 路径中的模型
        try:
            best_files = (glob.glob(os.path.join(CHECKPOINT_DIR, "classification_best_*.pth.tar"))
                         if ctrl.task_type == 'interaction'
                         else glob.glob(os.path.join(CHECKPOINT_DIR, "regression_best_*.pth.tar")))
            time_threshold = time.time() - (24 * 60 * 60)  # 24小时内的模型
            for f in best_files:
                try:
                    mod_time = os.path.getmtime(f)
                    if mod_time > time_threshold:
                        model_options.append((f' From Drive: {os.path.basename(f)}', f))
                except:
                    pass
        except:
            pass


        model_selector = widgets.Dropdown(
            options=model_options,
            value='pretrained',
            description='Select Model:',
            disabled=False,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='550px')
        )

        # 自定义 Drive 路径输入（初始隐藏）
        drive_path_input = widgets.Text(
            value='',
            placeholder='/content/drive/MyDrive/your_model.pth.tar',
            description='Drive path:',
            layout=widgets.Layout(width='550px', display='none'),
            style={'description_width': 'initial'}
        )

        # 上传模型的组件（初始隐藏）
        model_uploader = widgets.FileUpload(
            accept='.pth,.pth.tar',
            multiple=False,
            description='Upload model:',
            layout=widgets.Layout(width='500px', display='none')
        )
        default_desc = widgets.HTML("<p><a href= 'https://github.com/caixiaoyao2025/scientific_training1_cxy/blob/main/architecture.py' target='_blank'>click to view the architecture of the default model</a></p>")
        cp_reg = widgets.HTML("<p>checkpoints reserved at https://github.com/caixiaoyao2025/scientific_training1_cxy/blob/main/regression_model.pth</p>")
        cp_cla = widgets.HTML("<p>checkpoints reserved at https://github.com/caixiaoyao2025/scientific_training1_cxy/blob/main/simpledotproductclassifier.pth</p>")
        if ctrl.task_type == 'regression':
          cp_reg.layout.display = 'block'
          cp_cla.layout.display = 'none'
        else:
          cp_reg.layout.display = 'none'
          cp_cla.layout.display = 'block'
        default_desc.layout.display = 'block'
        def on_model_select(change):
            if change['new'] == 'pretrained':
              default_desc.layout.display = 'block'
              if ctrl.task_type == 'regression':
                cp_reg.layout.display = 'block'
                cp_cla.layout.display = 'none'
              else:
                cp_reg.layout.display = 'none'
                cp_cla.layout.display = 'block'
            else:
              cp_reg.layout.display = 'none'
              cp_cla.layout.display = 'none'
              default_desc.layout.display = 'none'
        model_selector.observe(on_model_select, names='value')

        # 任务选择
        task_selector = widgets.Dropdown(
            options=[
                ('Single Pair Prediction', 'case1'),
                ('Batch Prediction', 'case4')
            ],
            value=None,
            description='Select Prediction Task:',
            disabled=False,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )

        dynamic_input_area = widgets.Output()
        prediction_output = widgets.Output(layout=widgets.Layout(height='500px', overflow_y='auto'))

        # 开始预测按钮（初始禁用）
        start_predict_btn = widgets.Button(
            description="🚀 Start Prediction",
            button_style='success',
            icon='play',
            layout=widgets.Layout(width='200px', height='40px', margin='20px'),
            disabled=True
        )

        def on_task_change(change):
            with dynamic_input_area:
                clear_output()

                if not change['new']:
                    start_predict_btn.disabled = True
                    return

                current_task = change['new']
                start_predict_btn.disabled = False

                # case1: Single Pair Prediction
                if current_task == 'case1':
                    desc = widgets.HTML("<h4>Single Pair Prediction</h4><p>Predict interaction for a single protein-molecule pair</p>")

                    pro_input = widgets.Textarea(
                        description='Protein Sequence:',
                        placeholder='Enter protein sequence...',
                        layout=widgets.Layout(width='90%', height='100px'),
                        style={'description_width': 'initial'}
                    )

                    drug_input = widgets.Text(
                        description='Molecule SMILES:',
                        placeholder='Enter SMILES string...',
                        layout=widgets.Layout(width='90%'),
                        style={'description_width': 'initial'}
                    )

                    global current_inputs
                    current_inputs = {
                        'type': 'case1',
                        'pro_input': pro_input,
                        'drug_input': drug_input
                    }

                    display(widgets.VBox([desc, pro_input, drug_input]))

                # case4: Batch Prediction
                elif current_task == 'case4':
                    desc = widgets.HTML("<h4>Batch Prediction</h4><p>Upload CSV with multiple protein-molecule pairs</p>")

                    file_upload = widgets.FileUpload(
                        accept='.csv',
                        multiple=False,
                        description='Upload CSV:',
                        layout=widgets.Layout(width='90%')
                    )

                    sample_link = widgets.HTML(
                        value='<a href="https://github.com/caixiaoyao2025/scientific_training1_cxy/blob/main/examples.csv" target="_blank">Download Example Data</a>'
                    )

                    current_inputs = {
                        'type': 'case4',
                        'file_upload': file_upload
                    }

                    display(widgets.VBox([desc, file_upload, sample_link]))

        task_selector.observe(on_task_change, names='value')

        def on_start_predict(b):
            with prediction_output:
                clear_output()

                print("=" * 50)
                print("🚀 Starting deterministic prediction...")
                print("=" * 50)

                # 再次固定随机种子
                set_seed(42)

                try:
                    # 1. Load protein encoder
                    print("\n1. Loading protein encoder...")
                    ctrl.tokenizer_pro = AutoTokenizer.from_pretrained(
                        "westlake-repl/SaProt_35M_AF2_seqOnly",
                        revision="6628b8a5288695045fdb4575a361e84fdfefe5c8",
                        trust_remote_code=True
                    )
                    ctrl.model_pro = AutoModel.from_pretrained(
                        "westlake-repl/SaProt_35M_AF2_seqOnly",
                        revision="6628b8a5288695045fdb4575a361e84fdfefe5c8",
                        trust_remote_code=True
                    ).to(device)
                    ctrl.model_pro.eval()
                    print("   ✅ Done")

                    # 2. Load drug encoder
                    print("\n2. Loading drug encoder...")
                    ctrl.tokenizer_drug = AutoTokenizer.from_pretrained(
                        "DeepChem/ChemBERTa-77M-MLM",
                        revision="ed8a5374f2024ec8da53760af91a33fb8f6a15ff",
                        trust_remote_code=True
                    )
                    ctrl.model_drug = AutoModel.from_pretrained(
                        "DeepChem/ChemBERTa-77M-MLM",
                        revision="ed8a5374f2024ec8da53760af91a33fb8f6a15ff",
                        trust_remote_code=True
                    ).to(device)
                    ctrl.model_drug.eval()
                    print("   ✅ Done")

                    # 3. Load prediction model - 根据用户选择的模型加载
                    print("\n3. Loading prediction model...")
                    selected_model = model_selector.value

                    checkpoint = None

                    if selected_model == 'pretrained':
                        # 使用预训练模型
                        ctrl.current_model = load_prediction_model_deterministic(ctrl.task_type, device)
                        print("   Using pre-trained model")
                        checkpoint = None

                    elif selected_model == 'upload':
                        # 从本地上传的模型
                        if not model_uploader.value:
                            print("   ❌ Please upload a model file first")
                            return
                        content = list(model_uploader.value.values())[0]['content']
                        checkpoint = torch.load(io.BytesIO(content), map_location='cpu', weights_only=False)
                        print("   Using uploaded model")

                    elif selected_model == 'custom_drive':
                        # 从自定义 Drive 路径加载模型
                        custom_path = drive_path_input.value.strip()
                        if not custom_path:
                            print("   ❌ Please enter a valid Drive path")
                            return
                        if not os.path.exists(custom_path):
                            print(f"   ❌ File not found: {custom_path}")
                            print("   Make sure you have mounted Google Drive and the path is correct")
                            return
                        checkpoint = torch.load(custom_path, map_location='cpu', weights_only=False)
                        print(f"   Loading model from Drive: {custom_path}")

                    else:
                        # 从默认 Drive 路径加载训练好的模型
                        print(f"   Loading model from: {selected_model}")
                        checkpoint = torch.load(selected_model, map_location='cpu', weights_only=False)

                    # 如果有 checkpoint，加载模型权重
                    if checkpoint is not None:
                        if ctrl.task_type == 'interaction':
                            ctrl.current_model = InteractionModel(
                                checkpoint.get('pro_dim', 480),
                                checkpoint.get('drug_dim', 384)
                            ).to(device)
                        else:
                            ctrl.current_model = MyModel(
                                checkpoint.get('pro_dim', 480),
                                checkpoint.get('drug_dim', 384)
                            ).to(device)

                        # 加载模型权重
                        if 'model_state' in checkpoint:
                            ctrl.current_model.load_state_dict(checkpoint['model_state'])
                        elif 'model_state_dict' in checkpoint:
                            ctrl.current_model.load_state_dict(checkpoint['model_state_dict'])
                        else:
                            ctrl.current_model.load_state_dict(checkpoint)

                        ctrl.current_model.eval()
                        print("   ✅ Model loaded successfully")

                    print("\n✅ All models loaded successfully!")

                    # Get input based on task type
                    task_type = current_inputs['type']

                    if task_type == 'case1':
                        # Single Pair Prediction
                        pro = current_inputs['pro_input'].value.strip()
                        drug = current_inputs['drug_input'].value.strip()

                        if not pro or not drug:
                            print("\n❌ Please enter both protein sequence and molecule SMILES")
                            return
                        clear_output(wait=True)
                        print(f"\nProtein: {pro[:50]}..." if len(pro) > 50 else f"\nProtein: {pro}")
                        print(f"Molecule: {drug}")

                        # 使用确定性预测（返回标量）
                        score = run_deterministic_prediction(
                            [pro], [drug], ctrl.current_model,
                            ctrl.tokenizer_pro, ctrl.model_pro,
                            ctrl.tokenizer_drug, ctrl.model_drug,
                            ctrl.task_type
                        )

                        print(f"\n📊 Prediction result: {score:.6f}")
                        if ctrl.task_type == 'interaction':
                            print("(0-1 scale, >0.5 means likely to interact, higher = stronger interaction)")
                        else:
                            print("KIBA scores combine the information from different sources, including Kd, Ki (inhibitor constant), and IC50 (the concentration required to produce half-maximum inhibition) into a single bioactivity score.  KIBA affinity scores are ranged from 0.0 to 17.2. The lower the KIBA score the stronger the binding affinity between the drug and the target.")

                    elif task_type == 'case4':
                        # Batch Prediction
                        if not current_inputs['file_upload'].value:
                            print("\n❌ Please upload a file")
                            return

                        content = list(current_inputs['file_upload'].value.values())[0]['content']
                        df = pd.read_csv(io.BytesIO(content))

                        # Identify columns
                        pro_col = None
                        for col in ['seq', 'sequence', 'protein', 'Protein']:
                            if col in df.columns:
                                pro_col = col
                                break

                        drug_col = None
                        for col in ['smiles', 'SMILES', 'drug', 'Drug', 'compound', 'Compound']:
                            if col in df.columns:
                                drug_col = col
                                break

                        if pro_col is None or drug_col is None:
                            print("\n❌ CSV must contain protein and SMILES columns")
                            print(f"   Found columns: {df.columns.tolist()}")
                            return

                        n = len(df)
                        print(f"\nProcessing {n} pairs...")
                        print(f"   Protein column: {pro_col}")
                        print(f"   SMILES column: {drug_col}")

                        # 使用确定性预测（返回数组）
                        scores = run_deterministic_prediction(
                            df[pro_col].tolist(), df[drug_col].tolist(),
                            ctrl.current_model,
                            ctrl.tokenizer_pro, ctrl.model_pro,
                            ctrl.tokenizer_drug, ctrl.model_drug,
                            ctrl.task_type
                        )

                        df['Predicted_result'] = scores
                        df = df.sort_values(by='Predicted_result', ascending=False)

                        # 显示前10个结果
                        clear_output(wait=True)
                        print(f"\n📊 Top 10 predictions:")
                        print(df[[pro_col, drug_col, 'Predicted_result']].head(10).to_string(index=False))

                        # Save results
                        csv_filename = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
                        df.to_csv(csv_filename, index=False)
                        print(f"\n✅ Full results saved to: {csv_filename}")

                        if ctrl.task_type == 'interaction':
                            print("\n(0-1 scale, >0.5 means likely to interact)")
                        else:
                            print("\n(Lower score = stronger binding affinity)")

                        if IN_COLAB:
                            from google.colab import files
                            btn_download = widgets.Button(
                                description="📥 Download prediction results",
                                button_style='success',
                                icon='file-download',
                                layout=widgets.Layout(width='250px')
                            )
                            def on_download_click(b):
                                files.download(csv_filename)
                            btn_download.on_click(on_download_click)
                            display(btn_download)

                except Exception as e:
                    print(f"\n❌ Error: {str(e)}")
                    traceback.print_exc()

        start_predict_btn.on_click(on_start_predict)

        main_ui = widgets.VBox([
            title,
            widgets.HTML("<hr>"),
            model_selector,
            default_desc,
            cp_cla,
            cp_reg,
            widgets.HTML("<hr>"),
            task_selector,
            dynamic_input_area,
            widgets.HBox([start_predict_btn], layout=widgets.Layout(justify_content='center')),
            widgets.HTML("<hr>"),
            widgets.HTML("<h4>Results:</h4>"),
            prediction_output,
        ], layout=widgets.Layout(padding='20px'))

        display(main_ui)
# Bind button events
back_btn.on_click(lambda b: jump(b, next=show_main_menu))
refresh_btn.on_click(lambda b: jump(b, next=None))
stop_btn.on_click(stop_thread)

clear_output()
print("\n" + "="*60)

show_main_menu()
display(main_display)
display(*system_widgets)

# Main polling loop
with ui_events() as poll:
    try:
        while True:
            poll(10)
            if ctrl.is_training_finished and not ctrl.stop_requested:
                show_post_training_ui()
                ctrl.is_training_finished = False
            time.sleep(0.1)
    except KeyboardInterrupt:
        print("KeyboardInterrupt")
    except Exception as e:
        print(f"Polling exception: {e}")
    finally:
        stop_thread(None)
        clear_output()
        end_hint = HTML("<h2>Program interrupted, please click the run button to restart.</h2>")
        display(end_hint)